In [11]:
#SKLEARN MODELS
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv("cleaned_dataset.csv")

X = df.drop(["target", "id", "painting", "feel_text", "food", "soundtrack"], axis=1)
y = df["target"]

# get all numeric columns after initial preprocessing
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

# null values in original text col
X["feel_text"] = df["feel_text"].fillna("")
X["food"] = df["food"].fillna("")
X["soundtrack"] = df["soundtrack"].fillna("")

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("imputer", SimpleImputer(strategy="mean")), ("scaler", StandardScaler())]), numeric_cols),
        ("feel_txt", TfidfVectorizer(max_features=500, min_df=3), "feel_text"),
        ("food_txt", TfidfVectorizer(max_features=500, min_df=3), "food"),
        ("soundtrack_txt", TfidfVectorizer(max_features=500, min_df=3), "soundtrack"),
    ],
    remainder="drop"
)

# numeric-only branch for targeted ablation checks
numeric_only_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("imputer", SimpleImputer(strategy="mean")), ("scaler", StandardScaler())]), numeric_cols),
    ],
    remainder="drop"
)

# text-only preprocess branch for BernoulliNB
text_preprocess = ColumnTransformer(
    transformers=[
        ("feel_bin", CountVectorizer(max_features=500, min_df=3, binary=True), "feel_text"),
        ("food_bin", CountVectorizer(max_features=500, min_df=3, binary=True), "food"),
        ("soundtrack_bin", CountVectorizer(max_features=500, min_df=3, binary=True), "soundtrack"),
    ],
    remainder="drop"
)

base_models = {
    "log_reg": LogisticRegression(max_iter=2000),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "knn": KNeighborsClassifier(),
    "bernoulli_nb": BernoulliNB()
}

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)
print(f"Split sizes: train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")


def evaluate_model(model):
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    return accuracy_score(y_val, preds), f1_score(y_val, preds, average="macro")


def build_pair_stack(model_name, model, full_preprocess):
    return StackingClassifier(
        estimators=[
            ("bern_text", Pipeline([
                ("preprocess", text_preprocess),
                ("model", BernoulliNB())
            ])),
            (f"{model_name}_full", Pipeline([
                ("preprocess", full_preprocess),
                ("model", model)
            ])),
        ],
        final_estimator=LogisticRegression(max_iter=2000),
        stack_method="predict_proba",
        cv=5
    )


# base models
base_scores = []
for name, model in base_models.items():
    acc, f1m = evaluate_model(model)
    base_scores.append({"model": name, "accuracy": acc, "f1_macro": f1m})
    print(name, acc)

# text-only Bernoulli model
bernoulli_text_pipe = Pipeline([("preprocess", text_preprocess), ("model", BernoulliNB())])
bernoulli_text_pipe.fit(X_train, y_train)
bernoulli_text_preds = bernoulli_text_pipe.predict(X_val)
print("bernoulli_text", accuracy_score(y_val, bernoulli_text_preds))

# stacked ensemble (all models)
stacked_model = StackingClassifier(
    estimators=[
        ("bern_text", Pipeline([
            ("preprocess", text_preprocess),
            ("model", BernoulliNB())
        ])),
        ("log_full", Pipeline([
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=2000))
        ])),
        ("rf_full", Pipeline([
            ("preprocess", preprocess),
            ("model", RandomForestClassifier(n_estimators=300, random_state=42))
        ])),
        ("knn_full", Pipeline([
            ("preprocess", preprocess),
            ("model", KNeighborsClassifier())
        ])),
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    stack_method="predict_proba",
    cv=5
)
stacked_model.fit(X_train, y_train)
stacked_preds = stacked_model.predict(X_val)
print("stacked_ensemble", accuracy_score(y_val, stacked_preds))

# Bernoulli text + one model
pair_results = []
pair_predictions = {}
pair_candidates = {k: v for k, v in base_models.items() if k != "bernoulli_nb"}

for pair_name, pair_model in pair_candidates.items():
    pair_stack = build_pair_stack(pair_name, pair_model, preprocess)
    pair_stack.fit(X_train, y_train)
    pair_preds = pair_stack.predict(X_val)
    pair_predictions[pair_name] = pair_preds

    pair_results.append({
        "pair_model": f"bernoulli_text + {pair_name}",
        "accuracy": accuracy_score(y_val, pair_preds),
        "f1_macro": f1_score(y_val, pair_preds, average="macro")
    })

print("\nPair (Bernoulli + one model)")
display(pd.DataFrame(pair_results).sort_values(by=["accuracy", "f1_macro"], ascending=False))

logreg_pair_experiments = []
logreg_pair_predictions = {}

for variant_name, variant_preprocess in [("log_reg_full", preprocess), ("log_reg_numeric_only", numeric_only_preprocess)]:
    exp_stack = build_pair_stack("log_reg", LogisticRegression(max_iter=2000), variant_preprocess)
    exp_stack.fit(X_train, y_train)
    exp_preds = exp_stack.predict(X_val)
    logreg_pair_predictions[variant_name] = exp_preds

    logreg_pair_experiments.append({
        "variant": f"bernoulli_text + {variant_name}",
        "accuracy": accuracy_score(y_val, exp_preds),
        "f1_macro": f1_score(y_val, exp_preds, average="macro"),
    })

print("\nBernoulli + log_reg experiment")
display(pd.DataFrame(logreg_pair_experiments).sort_values(by=["accuracy", "f1_macro"], ascending=False))

full_vs_numeric_diff = float((
    logreg_pair_predictions["log_reg_full"] != logreg_pair_predictions["log_reg_numeric_only"]
).mean())
print("Prediction disagreement (full vs numeric-only):", round(full_vs_numeric_diff, 4))

# difference matrix between pair-stacks
pair_names = list(pair_predictions.keys())
diff_matrix = pd.DataFrame(index=pair_names, columns=pair_names, dtype=float)
for a in pair_names:
    for b in pair_names:
        diff_matrix.loc[a, b] = round(float((pair_predictions[a] != pair_predictions[b]).mean()), 4)

print("\nSamples where predictions differ")
display(diff_matrix)


Split sizes: train: 966, val: 322, test: 322
log_reg 0.9099378881987578
random_forest 0.9254658385093167
decision_tree 0.7981366459627329
knn 0.8105590062111802
bernoulli_nb 0.8975155279503105
bernoulli_text 0.8043478260869565
stacked_ensemble 0.9409937888198758

Pair (Bernoulli + one model)


,pair_model,accuracy,f1_macro
0,bernoulli_text + log_reg,0.934783,0.934498
1,bernoulli_text + random_forest,0.925466,0.925007
3,bernoulli_text + knn,0.897516,0.897624
2,bernoulli_text + decision_tree,0.847826,0.844530



Bernoulli + log_reg experiment


,variant,accuracy,f1_macro
0,bernoulli_text + log_reg_full,0.934783,0.934498
1,bernoulli_text + log_reg_numeric_only,0.922360,0.921783


Prediction disagreement (full vs numeric-only): 0.0342

Samples where predictions differ


,log_reg,random_forest,decision_tree,knn
log_reg,0.0000,0.0652,0.1273,0.0776
random_forest,0.0652,0.0000,0.1087,0.0683
decision_tree,0.1273,0.1087,0.0000,0.1242
knn,0.0776,0.0683,0.1242,0.0000


In [ ]:
# Export params logreg param
import json
import numpy as np

# Refit a dedicated log_reg pipeline so exported parameters are self-contained
logreg_export_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])
logreg_export_pipe.fit(X_train_val, y_train_val)

pre_fitted = logreg_export_pipe.named_steps["preprocess"]
lr_fitted = logreg_export_pipe.named_steps["model"]

num_pipe = pre_fitted.named_transformers_["num"]
imputer = num_pipe.named_steps["imputer"]
scaler = num_pipe.named_steps["scaler"]

feel_vec = pre_fitted.named_transformers_["feel_txt"]
food_vec = pre_fitted.named_transformers_["food_txt"]
soundtrack_vec = pre_fitted.named_transformers_["soundtrack_txt"]

np.savez(
    "logreg_params.npz",
    coef=np.array(lr_fitted.coef_, dtype=np.float64),
    intercept=np.array(lr_fitted.intercept_, dtype=np.float64),
    class_order=np.array(lr_fitted.classes_),
    numeric_cols=np.array(numeric_cols, dtype=object),
    num_means=np.array(imputer.statistics_, dtype=np.float64),
    num_stds=np.array(scaler.scale_, dtype=np.float64),
    train_indices=np.array(X_train.index),
    val_indices=np.array(X_val.index),
    train_val_indices=np.array(X_train_val.index),
    test_indices=np.array(X_test.index),
)

tfidf_meta = {
    "feel_text": {
        "vocab": {k: int(v) for k, v in feel_vec.vocabulary_.items()},
        "idf": [float(x) for x in feel_vec.idf_],
    },
    "food": {
        "vocab": {k: int(v) for k, v in food_vec.vocabulary_.items()},
        "idf": [float(x) for x in food_vec.idf_],
    },
    "soundtrack": {
        "vocab": {k: int(v) for k, v in soundtrack_vec.vocabulary_.items()},
        "idf": [float(x) for x in soundtrack_vec.idf_],
    },
}

with open("logreg_tfidf.json", "w", encoding="utf-8") as f:
    json.dump(tfidf_meta, f)

In [ ]:
# Export params for manual Bernoulli Naive Bayes
import json
import numpy as np

bernoulli_export_pipe = Pipeline([
    ("preprocess", text_preprocess),
    ("model", BernoulliNB()),
])
bernoulli_export_pipe.fit(X_train_val, y_train_val)

pre_fitted = bernoulli_export_pipe.named_steps["preprocess"]
nb_fitted = bernoulli_export_pipe.named_steps["model"]

feel_vec = pre_fitted.named_transformers_["feel_bin"]
food_vec = pre_fitted.named_transformers_["food_bin"]
soundtrack_vec = pre_fitted.named_transformers_["soundtrack_bin"]

feature_log_prob = np.asarray(nb_fitted.feature_log_prob_, dtype=np.float64)
neg_log_prob = np.log(np.clip(1.0 - np.exp(feature_log_prob), 1e-12, 1.0))

np.savez(
    "bernoulli_params.npz",
    feature_log_prob=feature_log_prob,
    neg_log_prob=neg_log_prob,
    class_log_prior=np.asarray(nb_fitted.class_log_prior_, dtype=np.float64),
    class_order=np.asarray(nb_fitted.classes_),
)

bernoulli_vocab = {
    "feel_text": {
        "vocab": {k: int(v) for k, v in feel_vec.vocabulary_.items()},
        "size": len(feel_vec.vocabulary_),
        "offset": 0,
    },
    "food": {
        "vocab": {k: int(v) for k, v in food_vec.vocabulary_.items()},
        "size": len(food_vec.vocabulary_),
        "offset": len(feel_vec.vocabulary_),
    },
    "soundtrack": {
        "vocab": {k: int(v) for k, v in soundtrack_vec.vocabulary_.items()},
        "size": len(soundtrack_vec.vocabulary_),
        "offset": len(feel_vec.vocabulary_) + len(food_vec.vocabulary_),
    },
}

with open("bernoulli_vocab.json", "w", encoding="utf-8") as f:
    json.dump(bernoulli_vocab, f)


In [ ]:
# Export stacked meta-model params
STACK_META_PATH = "stack_meta_params.npz"

nb_train_proba = bernoulli_export_pipe.predict_proba(X_train_val)
lr_train_proba = logreg_export_pipe.predict_proba(X_train_val)
stack_train_X = np.hstack([nb_train_proba, lr_train_proba])

stack_meta_model = LogisticRegression(max_iter=2000)
stack_meta_model.fit(stack_train_X, y_train_val)

np.savez(
    STACK_META_PATH,
    coef=np.asarray(stack_meta_model.coef_, dtype=np.float64),
    intercept=np.asarray(stack_meta_model.intercept_, dtype=np.float64),
    class_order=np.asarray(stack_meta_model.classes_),
    bernoulli_class_order=np.asarray(bernoulli_export_pipe.classes_),
    logreg_class_order=np.asarray(logreg_export_pipe.classes_),
    base_model_names=np.array(["bernoulli_nb", "logreg"], dtype=object),
)

In [15]:
sklearn_train_proba = logreg_export_pipe.predict_proba(X_train_val)
sklearn_test_proba = logreg_export_pipe.predict_proba(X_test)

np.savez(
    "logreg_sklearn_benchmark.npz",
    sklearn_train_proba=np.array(sklearn_train_proba, dtype=np.float64),
    sklearn_test_proba=np.array(sklearn_test_proba, dtype=np.float64),
    class_order=np.array(lr_fitted.classes_),
    train_indices=np.array(X_train_val.index),
    test_indices=np.array(X_test.index),
)

In [16]:
preprocessor_fitted = preprocess.fit(X_train_val, y_train_val)
processed_feature_names = preprocessor_fitted.get_feature_names_out()

print("\nFeature Importances/Coefficients")

for name, model in base_models.items():
    print(f"\nModel: {name}")
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train_val, y_train_val)

    if hasattr(model, 'coef_'):
        if model.coef_.ndim > 1:
            avg_abs_coef = abs(model.coef_).mean(axis=0)
        else:
            avg_abs_coef = abs(model.coef_)

        coef_df = pd.DataFrame({'Feature': processed_feature_names, 'Coefficient': avg_abs_coef})
        coef_df = coef_df.sort_values(by='Coefficient', ascending=False)
        print("Top 10 most important features:")
        display(coef_df.head(10))

    elif hasattr(model, 'feature_importances_'):
        importance_df = pd.DataFrame({'Feature': processed_feature_names, 'Importance': model.feature_importances_})
        importance_df = importance_df.sort_values(by='Importance', ascending=False)
        print("Top 10 most important features:")
        display(importance_df.head(10))


Feature Importances/Coefficients

Model: log_reg
Top 10 most important features:


,Feature,Coefficient
755,food_txt__salad,1.107455
470,feel_txt__time,1.075784
563,food_txt__blueberry,0.990412
323,feel_txt__night,0.827059
408,feel_txt__sky,0.782440
68,feel_txt__awe,0.698557
746,food_txt__ramen,0.656287
36,num__season_Winter,0.641025
1244,soundtrack_txt__strong,0.634642
146,feel_txt__dizzy,0.617077



Model: random_forest
Top 10 most important features:


,Feature,Importance
2,num__content,0.051974
34,num__season_Spring,0.051832
4,num__uneasy,0.050900
6,num__n_objects,0.040857
1,num__sombre,0.039243
36,num__season_Winter,0.035078
3,num__calm,0.034091
33,num__season_Fall,0.025824
470,feel_txt__time,0.023956
5,num__n_colors,0.022162



Model: decision_tree
Top 10 most important features:


,Feature,Importance
4,num__uneasy,0.277175
34,num__season_Spring,0.145521
36,num__season_Winter,0.082200
6,num__n_objects,0.077314
2,num__content,0.040031
408,feel_txt__sky,0.020579
470,feel_txt__time,0.019086
1,num__sombre,0.014018
7,num__pay,0.013933
18,num__feel_text_avg_word_len,0.013377



Model: knn

Model: bernoulli_nb
